In [ ]:
from config_dirs import OUTPUT_1_1_PMIDS
from Bio import Entrez
import pandas as pd
import time

# Set your email address (NCBI requires it)
Entrez.email = 'example@gmail.com'

# Define the base search query with the human organism filter
base_search_query = (
    '("vascular diseases"[MeSH Terms] OR "microvascular disorders"[MeSH Terms] OR '
    '"microangiopathy"[MeSH Terms] OR "capillaropathy"[MeSH Terms] OR '
    '"vessel disease*"[tiab] OR "microvasc*"[tiab] OR "microvessel*"[tiab] OR '
    '"microangiopat*"[tiab] OR "capillaropat*"[tiab]) AND '
    '("genes"[MeSH Terms] OR "genetic predisposition to disease"[MeSH Terms] OR '
    '"genome"[MeSH Terms] OR "mutation"[MeSH Terms] OR gene[tiab] OR genetic[tiab] OR '
    'genome[tiab] OR mutation[tiab]) AND '
    '"humans"[MeSH Terms]'
)

# Function to fetch and save records for a given year
def fetch_and_save_records_for_year(year):
    print(f"\nProcessing year: {year}")
    # Update the search query to include the publication date
    search_query = f'{base_search_query} AND ("{year}"[PDAT])'

    # Use esearch to get PMIDs for the current year
    try:
        handle = Entrez.esearch(db="pubmed", term=search_query, retmax=10000)
        search_results = Entrez.read(handle)
        handle.close()
    except Exception as e:
        print(f"An error occurred during esearch for year {year}: {e}")
        return

    pmid_list = search_results.get('IdList', [])
    count = int(search_results.get('Count', 0))
    print(f"Number of publications found for {year}: {count}")

    if count == 0 or not pmid_list:
        print(f"No publications found for year {year}.")
        return

    # Initialize lists to store data for the current year
    pmids = []
    journal_names = []
    article_titles = []
    publication_years = []

    # Process PMIDs in batches for efetch
    batch_size = 200  # Max number of IDs per efetch request

    for start in range(0, len(pmid_list), batch_size):
        end = min(len(pmid_list), start + batch_size)
        print(f"Fetching records {start + 1} to {end} for year {year}...")
        id_batch = pmid_list[start:end]
        # Validate IDs
        id_batch = [pmid for pmid in id_batch if pmid.isdigit()]
        if not id_batch:
            print(f"No valid PMIDs in batch {start + 1} to {end} for year {year}.")
            continue  # Skip to the next batch
        ids = ','.join(id_batch)
        # Print IDs being fetched for debugging
        print(f"IDs being fetched: {ids}")

        try:
            # Use efetch to retrieve the records in PubMed XML format
            fetch_handle = Entrez.efetch(
                db="pubmed",
                id=ids,
                rettype="medline",
                retmode="xml"
            )
            records = Entrez.read(fetch_handle)
            fetch_handle.close()
        except Exception as e:
            print(f"An error occurred while fetching records for IDs: {ids}")
            print(f"Error message: {e}")
            continue  # Skip to the next batch

        # Extract information from each record
        for article in records.get('PubmedArticle', []):
            # PMID
            pmid = article['MedlineCitation']['PMID']

            # Journal Name
            journal_info = article['MedlineCitation']['Article']['Journal']
            journal_title = journal_info.get('Title', '')

            # Article Title
            article_title = article['MedlineCitation']['Article'].get('ArticleTitle', '')

            # Publication Year
            pub_date = journal_info.get('JournalIssue', {}).get('PubDate', {})
            # PubDate can be a dict with 'Year' key, or 'MedlineDate' key (e.g., '1975 Jan-Feb')
            if 'Year' in pub_date:
                pub_year = pub_date['Year']
            elif 'MedlineDate' in pub_date:
                # Extract year from MedlineDate string
                pub_year = pub_date['MedlineDate'][:4]
            else:
                pub_year = ''

            # Append to lists
            pmids.append(pmid)
            journal_names.append(journal_title)
            article_titles.append(article_title)
            publication_years.append(pub_year)

        # Be respectful of NCBI's server load
        time.sleep(0.3)  # Limit to 3 requests per second

    if not pmids:
        print(f"No data collected for year {year}.")
        return

    # Create a pandas DataFrame with the results for the current year
    df_year = pd.DataFrame({
        'PMID': pmids,
        'Journal': journal_names,
        'Title': article_titles,
        'Year': publication_years
    })

    # Reset the index of the DataFrame
    df_year.reset_index(drop=True, inplace=True)

    # Optionally, save the DataFrame to a CSV file
    filename = f'pubmed_results_{year}.csv'
    df_year.to_csv(filename, index=False)
    print(f"Data for year {year} saved to {filename}.")

    # Clear the lists to free up memory
    del pmids, journal_names, article_titles, publication_years

    # Be respectful of NCBI's server load
    time.sleep(1)  # Optional delay between years

# Define the year range
start_year = 2000
end_year = 2024  # Include the year 2024

# Loop over each year and process records
for year in range(start_year, end_year + 1):
    fetch_and_save_records_for_year(year)


In [ ]:
import pandas as pd
import glob

# Get a list of all CSV files in the current directory matching the pattern
csv_files = glob.glob("pubmed_results_*.csv")

# Initialize an empty list to store individual dataframes
dfs = []

# Loop through each CSV file and load it into a dataframe
for file in csv_files:
    df = pd.read_csv(file)
    dfs.append(df)

# Concatenate all the dataframes into a single dataframe
merged_df = pd.concat(dfs, ignore_index=True)

# Save in a file
merged_df.to_csv(OUTPUT_1_1_PMIDS, index=False)

In [ ]:
merged_df.head()

merged_df.shape

# Count the number of duplicated rows
num_duplicates = merged_df.duplicated().sum()
num_duplicates


# Count the occurrences of each unique journal
journal_counts = merged_df['Journal'].value_counts().reset_index()

# Rename the columns for clarity
journal_counts.columns = ['Journal', 'Count']

pd.set_option('display.max_rows', None)
journal_counts.head()

# Take the first 10 rows and create a subset dataframe
subset = journal_counts.head(10)
subset

journal_counts.to_csv("journal_counts.csv", index=False)